# 01 · Training runs (one run per session is fine; every run is resumable)
Pick a `RUN_ID`, then run all cells. If Colab disconnects, reconnect and **run all cells again**. Training continues from the
latest checkpoint on Drive (saved every 100 steps). When a run finishes, it writes `adapter.chkpt` to the run's directory.

| ID | Method | Setting | Trainable | LR | Claim tested |
|---|---|---|---|---|---|
| R1 | DiaBlo | N=64 | 13.42M | 5e-4 (repo default) | C1 |
| R2 | LoRA | r=16 | 12.39M | 2e-4 | C1 |
| R3 | DiaBlo | N=32 | 26.84M | 5e-4 | C2 |
| R4 | DiaBlo | N=128 | 6.71M | 5e-4 | C2 |
| R6* | LoRA | r=16 | 12.39M | 5e-4 | fairness (same LR as DiaBlo) |
| R7* | DiaBlo | N=64 | 13.42M | 2e-4 | C3 (LR robustness) |
| R8* | LoRA | r=32 | 24.77M | 2e-4 | C1 at ~27M |
| R5* | R1 / R2 with seed 1 | | | | variance |

\* optional, if time allows. Order of priority: R1, R2, R3, R4, then R6/R7, then the rest.
Shared settings (from `scripts/train_gsm8k.sh`): 3 epochs, effective batch 16, linear schedule, 100 warmup steps, dropout 0.05, modules qkvud, fp16.

In [ ]:
# ── Common setup (run every session) ─────────────────────────────────────────
REPO_URL = "https://github.com/Mrazakos/DiaBlo.git"   # fork with the colab-repro branch
BRANCH   = "colab-repro"
MODEL    = "Qwen2.5-1.5B"                                       # short name or HF id
HF_ID    = "Qwen/Qwen2.5-1.5B"

import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/diablo"
os.makedirs(DRIVE, exist_ok=True)
DATA    = f"{DRIVE}/datasets/{MODEL}"     # tokenized data is tokenizer-specific
RESULTS = f"{DRIVE}/results"
EVALS   = f"{DRIVE}/eval"
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"   # keep downloads across sessions
os.environ["WANDB_MODE"] = "disabled"

REPO = "/content/DiaBlo"
if not os.path.exists(REPO):
    if os.path.exists(f"{DRIVE}/DiaBlo"):          # alternative: repo uploaded to Drive
        subprocess.run(["cp", "-r", f"{DRIVE}/DiaBlo", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
%cd {REPO}
!git log -1 --oneline 2>/dev/null || true
!pip -q install peft
print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
RUN_ID = "R1"   # <── change me
RUNS = {
  "R1":    dict(method="diablo", num_blocks=64,  lr=5e-4, seed=42),
  "R2":    dict(method="lora",   lora_r=16,      lr=2e-4, seed=42),
  "R3":    dict(method="diablo", num_blocks=32,  lr=5e-4, seed=42),
  "R4":    dict(method="diablo", num_blocks=128, lr=5e-4, seed=42),
  "R6":    dict(method="lora",   lora_r=16,      lr=5e-4, seed=42),
  "R7":    dict(method="diablo", num_blocks=64,  lr=2e-4, seed=42),
  "R8":    dict(method="lora",   lora_r=32,      lr=2e-4, seed=42),
  "R5a":   dict(method="diablo", num_blocks=64,  lr=5e-4, seed=1),
  "R5b":   dict(method="lora",   lora_r=16,      lr=2e-4, seed=1),
}
# Keep identical across runs. Change only if the smoke test forced it (then re-run everything).
SHARED = dict(per_device_batch_size=4, gradient_accumulation_steps=4, max_epochs=3, warmup_steps=100,
              dropout=0.05, target_modules="qkvud", precision="fp16", save_steps=100)
cfg = {**SHARED, **RUNS[RUN_ID]}
flags = " ".join(f"--{k} {v}" for k, v in cfg.items())
CMD = f"python train.py --model_name {MODEL} --task GSM8k --data_dir {DATA} --output_dir {RESULTS} --run_name {RUN_ID} --resume {flags}"
print(CMD)

In [ ]:
# Logs are also appended to Drive, so you can watch the loss even after a disconnect.
!mkdir -p {RESULTS}/{RUN_ID}
!{CMD} 2>&1 | tee -a {RESULTS}/{RUN_ID}/train.log

In [ ]:
import os
done = os.path.exists(f"{RESULTS}/{RUN_ID}/log_history.json")
print("FINISHED" if done else "not finished: re-run this notebook to resume")
!ls -la {RESULTS}/{RUN_ID}

Tips for the free tier:
* Keep the browser tab open. Free sessions are cut after about 90 minutes idle and about 12 hours in total, and the GPU can be revoked earlier.
* If you are out of GPU quota for the day, the next session resumes from the last 100-step checkpoint. At most 100 steps are lost.
* `--resume` with no checkpoint simply starts fresh, so the same command works for both the first and later sessions.